<a href="https://colab.research.google.com/github/maria-gigi/Research-Machine-Learning-2026/blob/main/Tratamento_de_Dados_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
import pandas as pd

In [ ]:
uploaded = files.upload()

In [ ]:
import os
import pandas as pd

# Dicionário para armazenar os dados consolidados
# A chave será (sujeito, atividade) para agrupar na mesma linha
dados_consolidados = {}

excluir = ['Cb1', 'Cb2', 'AF3', 'AF4']

for nome_arquivo in uploaded.keys():
    # 1. Carregar o arquivo
    df_temp = pd.read_csv(nome_arquivo, sep='\t')

    # 2. Extrair informações do nome do arquivo
    nome_base = os.path.splitext(nome_arquivo)[0]
    partes = nome_base.split('_')

    # ID Sujeito (1º + 3º termo) e Atividade (2º termo)
    id_sujeito = f"{partes[0]}_{partes[2]}"
    atividade = partes[1]
    chave_grupo = (id_sujeito, atividade)

    # 3. Lógica condicional por tipo de arquivo
    if 'iREA' in nome_base:
        col_valor = 'WeiDegREA'
        prefixo = 'REA_'
    elif 'Hubs' in nome_base:
        col_valor = 'HubSim'
        prefixo = 'Hubs_'
    else:
        continue # Pula arquivos que não se encaixam

    # 4. Limpeza e Seleção
    df_temp = df_temp[~df_temp['Label'].isin(excluir)]
    df_selecionado = df_temp[['Label', col_valor]].copy()

    # 5. Transposição (Pivot)
    df_selecionado['id'] = 0
    df_pivoted = df_selecionado.pivot(index='id', columns='Label', values=col_valor)
    df_pivoted.columns.name = None
    df_pivoted = df_pivoted.add_prefix(prefixo)

    # 6. Agrupamento por Sujeito e Atividade
    if chave_grupo not in dados_consolidados:
        dados_consolidados[chave_grupo] = df_pivoted
    else:
        # Se já existe, concatena as colunas (eixo 1)
        dados_consolidados[chave_grupo] = pd.concat([dados_consolidados[chave_grupo], df_pivoted], axis=1)

# 7. Transformar o dicionário na tabela final
lista_final = []
for (suj, ativ), df_linha in dados_consolidados.items():
    df_linha.insert(0, 'Sujeito', suj)
    df_linha['Atividade'] = ativ
    lista_final.append(df_linha)

df_consolidado_final = pd.concat(lista_final, ignore_index=True)

# Exibir informações da tabela final
print(f"Formato final: {df_consolidado_final.shape}")
display(df_consolidado_final.head())

Formato final: (52, 118)


,Sujeito,Hubs_C1,Hubs_C2,Hubs_C3,Hubs_C4,Hubs_C5,Hubs_C6,Hubs_CP1,Hubs_CP2,Hubs_CP3,...,REA_PO4,REA_PO7,REA_PO8,REA_POz,REA_Pz,REA_T7,REA_T8,REA_TP7,REA_TP8,Atividade
0,65_01,176,127,503,372,579,1142,283,182,400,...,725067,709471,743930,626388,229439,495917,539354,589195,608557,AC
1,65_02,265,270,577,107,646,130,454,279,1112,...,541269,480557,527734,501639,290968,249810,245168,327128,287699,AC
2,65_02,447,463,698,1057,519,1149,880,978,1234,...,419651,362819,392654,393505,240182,182261,251019,200962,226028,BL
3,65_01,156,265,352,577,653,1662,500,232,511,...,542979,529713,543837,517317,168495,240014,215396,369131,336188,EV
4,65_02,760,849,1702,1250,2234,1451,1031,830,2030,...,267322,261050,223071,261825,153697,101858,89304,179092,146648,EV


In [ ]:
from google.colab import files

# Nome do arquivo que será criado
nome_csv = 'Tabela_REA_Hubs.csv'

# Salva o DataFrame em CSV (usando ponto e vírgula como separador para facilitar abertura no Excel Brasil)
df_consolidado_final.to_csv(nome_csv, index=False, sep=';', encoding='utf-8-sig')

# Baixa o arquivo para o meu computador
files.download(nome_csv)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>